- **Name:** 21_delta_advanced
- **Author:** Shamas Imran
- **Desciption:** Advanced operations with Delta Lake
- **Date:** 19-Aug-2025
<!--
REVISION HISTORY
Version          Date        Author           Desciption
01           19-Aug-2025   Shamas Imran       Implemented upserts (merge) in Delta  
                                              Used time travel queries  
                                              Performed vacuum and optimize  
-->

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from delta.tables import DeltaTable

# ------------------------------------------------------------
# 1) Spark Session
# ------------------------------------------------------------
spark = (
    SparkSession.builder
        .appName("Delta_Advanced_Features")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .getOrCreate()
)


StatementMeta(, 670870d4-6bf0-4f01-9047-09640c0f3235, 4, Finished, Available, Finished)

In [3]:
# ------------------------------------------------------------
# 2) Sample Student DataFrame
# ------------------------------------------------------------
student_schema = StructType([
    StructField('StudentID', IntegerType(), False),
    StructField('StudentName', StringType(), True),
    StructField('StudentAge', IntegerType(), True)
])

student_data = [
    (1, "Alice", 34),
    (2, "Bob", 45),
    (3, "Charlie", 29),
    (4, "Shamas", 40)
]

df_student = spark.createDataFrame(student_data, student_schema)

# Delta table path
delta_path = "abfss://shamas_ws@onelake.dfs.fabric.microsoft.com/test_Lakehouse.Lakehouse/Tables"

# ------------------------------------------------------------
# 3) Create Delta Table
# ------------------------------------------------------------

df_student.write.mode("overwrite").saveAsTable("Student")
# Now df_student is saved as Delta table

StatementMeta(, 670870d4-6bf0-4f01-9047-09640c0f3235, 5, Finished, Available, Finished)

In [18]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("OptimizeDemo").getOrCreate()

data = [(i, f"Name_{i}") for i in range(1000)]
df = spark.createDataFrame(data, ["StudentID", "StudentName"])

# df = df.repartition(70)
df.write.mode("overwrite").saveAsTable("Student_New")


StatementMeta(, fb13f04b-7f82-4f11-a931-0d7a6168ddc2, 20, Finished, Available, Finished)

In [31]:
# List all files in Delta folder
delta_path = "abfss://shamas_ws@onelake.dfs.fabric.microsoft.com/test_Lakehouse.Lakehouse/Tables/student"
all_files = mssparkutils.fs.ls(delta_path)

# Filter only Parquet files
parquet_files = [f.path for f in all_files if f.path.endswith(".parquet")]

print(f"Number of files: {len(parquet_files)}")

StatementMeta(, fb13f04b-7f82-4f11-a931-0d7a6168ddc2, 33, Finished, Available, Finished)

Number of files: 1


In [20]:
%%sql

OPTIMIZE student_new

StatementMeta(, fb13f04b-7f82-4f11-a931-0d7a6168ddc2, 22, Finished, Available, Finished)

<Spark SQL result set with 1 rows and 2 fields>

In [0]:
                                                            # ------------------------------------------------------------
                                                            # 4) Time Travel Queries
                                                            # ------------------------------------------------------------

In [23]:
from delta.tables import DeltaTable
delta_path = "abfss://shamas_ws@onelake.dfs.fabric.microsoft.com/test_Lakehouse.Lakehouse/Tables/student"

# Show latest version
delta_table = DeltaTable.forPath(spark, delta_path)
delta_table.toDF().show()

StatementMeta(, fb13f04b-7f82-4f11-a931-0d7a6168ddc2, 25, Finished, Available, Finished)

+---------+-----------+----------+
|StudentID|StudentName|StudentAge|
+---------+-----------+----------+
|        4|     Shamas|        40|
|        1|      Alice|        34|
|        2|        Bob|        45|
|        3|    Charlie|        31|
|        5|     Faizan|        26|
+---------+-----------+----------+



In [30]:
# Get full history
full_history = delta_table.history(1000)  # large number to cover all versions
full_history.select("version", "timestamp", "operation", "operationParameters").show(truncate=False)


StatementMeta(, fb13f04b-7f82-4f11-a931-0d7a6168ddc2, 32, Finished, Available, Finished)

+-------+-----------------------+---------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp              |operation                        |operationParameters                                                                                                                                                                        |
+-------+-----------------------+---------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|6      |2025-10-31 18:30:35.867|VACUUM END                       |{status -> COMPLETED}                                                                                                                                                

In [13]:
from delta.tables import DeltaTable
# Example: Read data as it was at version 0
df_version0 = spark.read.format("delta").option("versionAsOf", 0).load(delta_path)
df_version0.show()
# Key Point: Time travel allows you to query historical snapshots of Delta table

StatementMeta(, 670870d4-6bf0-4f01-9047-09640c0f3235, 15, Finished, Available, Finished)

+---------+-----------+----------+
|StudentID|StudentName|StudentAge|
+---------+-----------+----------+
|        3|    Charlie|        29|
|        4|     Shamas|        40|
|        1|      Alice|        34|
|        2|        Bob|        45|
+---------+-----------+----------+



In [15]:
# ------------------------------------------------------------
# 5) Upserts / MERGE INTO (handling CDC)
# ------------------------------------------------------------
# Example new data to upsert (some new, some existing StudentID)
student_updates = [
    (3, "Charlie", 31),  # existing StudentID, age updated
    (5, "Faizan", 26),    # new StudentID
    (6, "Arham", 26)
]

df_updates = spark.createDataFrame(student_updates, student_schema)
delta_table = DeltaTable.forPath(spark, delta_path)

delta_table.alias("tgt").merge(
    df_updates.alias("src"),
    "tgt.StudentID = src.StudentID"
).whenMatchedUpdate(set = {
    "StudentName": "src.StudentName",
    "StudentAge": "src.StudentAge"
}).whenNotMatchedInsert(values = {
    "StudentID": "src.StudentID",
    "StudentName": "src.StudentName",
    "StudentAge": "src.StudentAge"
}).execute()

# Show updated table
delta_table.toDF().show()

StatementMeta(, 670870d4-6bf0-4f01-9047-09640c0f3235, 17, Finished, Available, Finished)

+---------+-----------+----------+
|StudentID|StudentName|StudentAge|
+---------+-----------+----------+
|        4|     Shamas|        40|
|        1|      Alice|        34|
|        2|        Bob|        45|
|        3|    Charlie|        31|
|        5|     Faizan|        26|
|        6|      Arham|        26|
+---------+-----------+----------+



In [24]:
delta_table.delete("StudentID = 6")
delta_table.toDF().show()

StatementMeta(, 670870d4-6bf0-4f01-9047-09640c0f3235, 26, Finished, Available, Finished)

+---------+-----------+----------+
|StudentID|StudentName|StudentAge|
+---------+-----------+----------+
|        4|     Shamas|        40|
|        1|      Alice|        34|
|        2|        Bob|        45|
|        3|    Charlie|        31|
|        5|     Faizan|        26|
+---------+-----------+----------+



In [27]:
# List all files in Delta folder
all_files = mssparkutils.fs.ls(delta_path)

# Filter only Parquet files
parquet_files = [f.path for f in all_files if f.path.endswith(".parquet")]

# Show all Parquet files
print("Parquet files in Delta folder:")
for file in parquet_files:
    print(file)

StatementMeta(, 670870d4-6bf0-4f01-9047-09640c0f3235, 29, Finished, Available, Finished)

Parquet files in Delta folder:
abfss://shamas_ws@onelake.dfs.fabric.microsoft.com/test_Lakehouse.Lakehouse/Tables/student/part-00000-3baee833-2fb4-4d4a-a6fb-90f65f53b129-c000.snappy.parquet
abfss://shamas_ws@onelake.dfs.fabric.microsoft.com/test_Lakehouse.Lakehouse/Tables/student/part-00000-4746c988-d33f-4269-a057-a97ee88e9076-c000.snappy.parquet
abfss://shamas_ws@onelake.dfs.fabric.microsoft.com/test_Lakehouse.Lakehouse/Tables/student/part-00000-598773da-29c0-4d89-97db-a2d226c2a237-c000.snappy.parquet
abfss://shamas_ws@onelake.dfs.fabric.microsoft.com/test_Lakehouse.Lakehouse/Tables/student/part-00000-83259400-5df9-4b1a-a722-b9858c37bd84-c000.snappy.parquet
abfss://shamas_ws@onelake.dfs.fabric.microsoft.com/test_Lakehouse.Lakehouse/Tables/student/part-00000-9a245035-7da4-49d5-861d-14fbed60efb0-c000.snappy.parquet
abfss://shamas_ws@onelake.dfs.fabric.microsoft.com/test_Lakehouse.Lakehouse/Tables/student/part-00000-ac6d3daf-6fe6-4274-9c5b-1ad66d20efe5-c000.snappy.parquet
abfss://shamas_

In [26]:
                                                    # ------------------------------------------------------------
                                                    # 6.1) OPTIMIZE
                                                    # ------------------------------------------------------------
# OPTIMIZE (Delta Lake only in Databricks) improves query performance by compacting small files
spark.sql("OPTIMIZE student")
# spark.sql("OPTIMIZE delta.`abfss://shamas_ws@onelake.dfs.fabric.microsoft.com/test_Lakehouse.Lakehouse/Tables/Student`")



# What is Z-Ordering
# How Z-Ordering Works
# When you run a Z-ORDER command, Fabric reorganizes the underlying Parquet files of the Delta table:
#     1) Reads the existing files.
#     2) Sorts records based on a Z-value computed from the specified columns.
#     3) Rewrites data files to cluster related records together.
# The Z-value (Z-order curve) converts multi-dimensional data into a single dimension while preserving spatial locality — meaning nearby values in multiple columns remain close in storage.

# spark.sql("""
#     OPTIMIZE delta.`/lakehouse/default/PatientData`
#     ZORDER BY (PatientId, EventDate)
# """)

StatementMeta(, 670870d4-6bf0-4f01-9047-09640c0f3235, 28, Finished, Available, Finished)

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,numFilesUpdatedWithoutRewrite:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesUpdatedWithoutRewrite:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemovedBreakdown:array<struct<reason:string,metrics:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>>>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,

In [29]:
                                                    # ------------------------------------------------------------
                                                    # 6.2) VACUUM
                                                    # ------------------------------------------------------------
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", False)
# VACUUM removes old files to clean up storage
spark.sql("VACUUM student RETAIN 1 HOURS")

StatementMeta(, fb13f04b-7f82-4f11-a931-0d7a6168ddc2, 31, Finished, Available, Finished)

DataFrame[path: string]

In [39]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# ------------------------------------------------------------
# 7) Schema Evolution
# ------------------------------------------------------------
# Adding a new column (StudentGrade) and enabling schema evolution
new_student_data = [
    (26, "Basit", "B", 'Lahore')
]

new_schema = StructType([
    StructField('StudentID', IntegerType(), False),
    StructField('StudentName', StringType(), True),
    StructField('StudentGrade', StringType(), True),
    StructField('StudentCity', StringType(), True)
])

df_new = spark.createDataFrame(new_student_data, new_schema)

df_new.write.format("delta").mode("append").option("mergeSchema", "true").save(delta_path)
# Key Point: mergeSchema=True allows adding new columns without breaking the table

spark.read.format("delta").load(delta_path).show()

StatementMeta(, fb13f04b-7f82-4f11-a931-0d7a6168ddc2, 41, Finished, Available, Finished)

+---------+-----------+----------+------------+-----------+
|StudentID|StudentName|StudentAge|StudentGrade|StudentCity|
+---------+-----------+----------+------------+-----------+
|       26|      Basit|      NULL|           B|     Lahore|
|       18|      Adeel|        28|           A|       NULL|
|        4|     Shamas|        40|        NULL|       NULL|
|        1|      Alice|        34|        NULL|       NULL|
|        2|        Bob|        45|        NULL|       NULL|
|        3|    Charlie|        31|        NULL|       NULL|
|        5|     Faizan|        26|        NULL|       NULL|
|       20|      Qasim|      NULL|           C|       NULL|
|       25|       Adil|      NULL|           B|       NULL|
|       19|       Asim|      NULL|           B|       NULL|
+---------+-----------+----------+------------+-----------+

